# DevMind AI — Dataset Discovery & Collection

This notebook searches Hugging Face for candidate datasets. **Inspect dataset cards, licenses, schemas, and sample quality before using data for training.** Search results are candidates, not endorsements.

In [ ]:
!pip -q install datasets huggingface_hub pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/DevMindAI')
RAW_DIR = PROJECT_DIR / 'datasets/raw'
META_DIR = PROJECT_DIR / 'datasets/metadata'
RAW_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

## 1. Search for candidate datasets

Search terms include debugging, code repair, bug fixing, programming errors, and code instruction.

In [ ]:
from huggingface_hub import list_datasets
import pandas as pd

search_terms = ['debugging', 'code repair', 'bug fixing', 'programming errors', 'code instruction']
rows = []
for term in search_terms:
    print(f'--- Search: {term} ---')
    try:
        results = list(list_datasets(search=term, limit=20))
        for ds in results:
            rows.append({'search_term': term, 'dataset_id': ds.id, 'downloads': getattr(ds, 'downloads', None), 'likes': getattr(ds, 'likes', None)})
    except Exception as exc:
        print('Search error:', exc)

candidate_df = pd.DataFrame(rows).drop_duplicates(subset=['dataset_id'])
candidate_df.sort_values(['downloads', 'likes'], ascending=False, na_position='last').head(50)

## 2. Inspect a candidate dataset card

Choose a dataset from the search results. Review its Hugging Face dataset page and license before downloading.

In [ ]:
# Replace with a candidate dataset ID after inspecting its card.
DATASET_ID = ''  # Example format: 'organization/dataset-name'

if DATASET_ID:
    from huggingface_hub import dataset_info
    info = dataset_info(DATASET_ID)
    print('Dataset:', info.id)
    print('Card/license metadata:', info.cardData)
    print('Tags:', info.tags)

## 3. Load and inspect a selected dataset

Only run after reviewing the dataset card, license, and terms.

In [ ]:
# Set DATASET_ID and CONFIG if required by the selected dataset.
DATASET_ID = ''
CONFIG = None

if DATASET_ID:
    from datasets import load_dataset
    dataset = load_dataset(DATASET_ID, CONFIG)
    print(dataset)
    split_name = next(iter(dataset.keys()))
    print('Sample:', dataset[split_name][0])
else:
    print('Set DATASET_ID after dataset review. No dataset downloaded yet.')

## 4. Record dataset provenance

Keep a metadata record for every dataset. Do not combine sources until licenses, schema, and quality are reviewed.

In [ ]:
import json
from datetime import datetime, timezone

metadata = {
    'dataset_id': DATASET_ID,
    'source_url': '',
    'license': 'REVIEW REQUIRED',
    'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
    'review_status': 'not reviewed',
    'notes': 'Record license, terms, schema, quality, and permitted use before training.'
}
metadata_path = META_DIR / 'dataset_candidate_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Metadata template saved:', metadata_path)

## Next steps

1. Review candidate dataset cards and licenses.
2. Select suitable sources.
3. Download selected data to `datasets/raw/`.
4. Inspect schema and sample quality.
5. Normalize examples into the DevMind AI format.
6. Deduplicate and create train/validation/test splits.